In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.pipeline_monitoring (
  run_date DATE,
  layer STRING,
  target_table STRING,
  status STRING,
  message STRING,
  logged_at TIMESTAMP
)
USING DELTA
""")


In [0]:
from pyspark.sql import functions as F

MONITOR_TABLE = "workspace.default.pipeline_monitoring"

def log_run(run_date, layer, target_table, status, message):
    df = (
        spark.createDataFrame([(run_date, layer, target_table, status, message)],
                              ["run_date", "layer", "target_table", "status", "message"])
        .withColumn("run_date", F.to_date("run_date"))
        .withColumn("logged_at", F.current_timestamp())
        .select("run_date", "layer", "target_table", "status", "message", "logged_at")
    )

    (df.write
       .format("delta")
       .mode("append")
       .option("mergeSchema", "true")
       .saveAsTable(MONITOR_TABLE)
    )
